In [0]:
# Check current locations in database
import sys
sys.path.append('/Workspace/Users/nikhita.nikki@gmail.com/databricks-lakebase-app-day-2')

import lakebase

print("=== Current Locations in Database ===")
locations = lakebase.run_query("""
SELECT location, COUNT(*) as count 
FROM weather_documents 
GROUP BY location
ORDER BY location
""")

for loc in locations:
    print(f"  • {loc['location']}: {loc['count']} documents")

print(f"\nTotal unique locations: {len(locations)}")

In [0]:
%pip install sentence-transformers sqlalchemy

"""
Ingest Weather Documents -> Vector Embeddings (Lakebase)

Reads normalized weather documents from Lakebase, chunks narrative text,
embeds each chunk using sentence-transformers/all-MiniLM-L6-v2, and writes
the resulting vectors into weather_embeddings using psycopg2.
"""

import os
from datetime import datetime, timezone

from psycopg2.extras import execute_values
from sentence_transformers import SentenceTransformer

import lakebase


WEATHER_TABLE_NAME = "weather_documents"
WEATHER_EMBEDDINGS_TABLE_NAME = "weather_embeddings"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIM = 384

CHUNK_SIZE = 800
CHUNK_OVERLAP = 100
BATCH_SIZE = 32

In [0]:
def ensure_embeddings_table():
    """Create pgvector extension and weather embedding table."""

    with lakebase.get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute("CREATE EXTENSION IF NOT EXISTS vector")

            cur.execute(
                f"""
                CREATE TABLE IF NOT EXISTS {WEATHER_EMBEDDINGS_TABLE_NAME} (
                    id TEXT PRIMARY KEY,
                    document_id TEXT NOT NULL
                        REFERENCES {WEATHER_TABLE_NAME}(id)
                        ON DELETE CASCADE,
                    chunk_index INTEGER NOT NULL,
                    chunk_text TEXT NOT NULL,
                    embedding VECTOR({EMBEDDING_DIM}) NOT NULL,
                    model_name TEXT NOT NULL,
                    created_at TIMESTAMPTZ NOT NULL DEFAULT now(),

                    UNIQUE (document_id, chunk_index)
                )
                """
            )

            conn.commit()

In [0]:
def get_unembedded_documents() -> list[dict]:
    """Return weather documents that do not have embeddings yet."""

    return lakebase.run_query(
        f"""
        SELECT
            d.id,
            d.location,
            d.source_type,
            d.headline,
            d.narrative_text
        FROM {WEATHER_TABLE_NAME} d
        WHERE d.narrative_text IS NOT NULL
          AND TRIM(d.narrative_text) != ''
          AND NOT EXISTS (
              SELECT 1
              FROM {WEATHER_EMBEDDINGS_TABLE_NAME} e
              WHERE e.document_id = d.id
          )
        ORDER BY d.synced_at ASC
        """
    )

In [0]:
def chunk_text(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> list[str]:
    """Split text into overlapping character-based chunks."""

    if not text:
        return []

    text = text.strip()

    if not text:
        return []

    chunks = []
    step = chunk_size - chunk_overlap

    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()

        if chunk:
            chunks.append(chunk)

        if start + chunk_size >= len(text):
            break

    return chunks

In [0]:
def build_chunks(documents: list[dict]) -> list[dict]:
    """Convert weather documents into chunk records."""

    chunks = []

    for document in documents:
        document_chunks = chunk_text(
            document["narrative_text"]
        )

        for chunk_index, chunk in enumerate(document_chunks):
            chunks.append(
                {
                    "id": f"{document['id']}_{chunk_index}",
                    "document_id": document["id"],
                    "chunk_index": chunk_index,
                    "chunk_text": chunk,
                }
            )

    return chunks

In [0]:
def load_embedding_model():
    """Load the sentence-transformer model."""

    os.environ["HF_HOME"] = "/tmp/.cache/huggingface"
    os.environ["TRANSFORMERS_CACHE"] = "/tmp/.cache/huggingface"
    os.environ["HF_HUB_CACHE"] = "/tmp/.cache/huggingface"

    print(f"Loading embedding model: {EMBEDDING_MODEL_NAME}")

    return SentenceTransformer(
        EMBEDDING_MODEL_NAME,
        cache_folder="/tmp/.cache/huggingface",
    )

In [0]:
def embed_chunks(
    model: SentenceTransformer,
    chunks: list[dict],
) -> list[dict]:
    """Generate embeddings for weather chunks in batches."""

    if not chunks:
        return []

    results = []

    for i in range(0, len(chunks), BATCH_SIZE):
        batch = chunks[i:i + BATCH_SIZE]

        texts = [
            item["chunk_text"]
            for item in batch
        ]

        vectors = model.encode(
            texts,
            show_progress_bar=False,
        )

        for item, vector in zip(batch, vectors):
            results.append(
                {
                    **item,
                    "embedding": vector.tolist(),
                }
            )

        print(
            f"Embedded "
            f"{min(i + BATCH_SIZE, len(chunks))}/"
            f"{len(chunks)} chunks"
        )

    return results

In [0]:
def insert_embeddings(rows: list[dict]) -> int:
    """Batch insert weather chunk embeddings into Lakebase."""

    if not rows:
        return 0

    insert_data = []

    for row in rows:
        vector_string = (
            "["
            + ",".join(str(float(x)) for x in row["embedding"])
            + "]"
        )

        insert_data.append(
            (
                row["id"],
                row["document_id"],
                row["chunk_index"],
                row["chunk_text"],
                vector_string,
                EMBEDDING_MODEL_NAME,
            )
        )

    sql = f"""
        INSERT INTO {WEATHER_EMBEDDINGS_TABLE_NAME} (
            id,
            document_id,
            chunk_index,
            chunk_text,
            embedding,
            model_name
        )
        VALUES %s
        ON CONFLICT (document_id, chunk_index)
        DO UPDATE SET
            chunk_text = EXCLUDED.chunk_text,
            embedding = EXCLUDED.embedding,
            model_name = EXCLUDED.model_name,
            created_at = now()
    """

    template = (
        "(%s, %s, %s, %s, %s::vector, %s)"
    )

    with lakebase.get_connection() as conn:
        with conn.cursor() as cur:
            execute_values(
                cur,
                sql,
                insert_data,
                template=template,
                page_size=100,
            )

            affected = cur.rowcount
            conn.commit()

    return affected

In [0]:
def main():
    print("Starting weather embedding ingestion...")

    ensure_embeddings_table()

    documents = get_unembedded_documents()

    print(
        f"Found {len(documents)} unembedded weather documents"
    )

    if not documents:
        print("Nothing to embed.")
        return

    chunks = build_chunks(documents)

    print(
        f"Created {len(chunks)} chunks "
        f"from {len(documents)} documents"
    )

    model = load_embedding_model()

    embedded_chunks = embed_chunks(
        model,
        chunks,
    )

    inserted = insert_embeddings(
        embedded_chunks
    )

    print(
        f"Inserted/updated {inserted} "
        f"weather embeddings"
    )


if __name__ == "__main__":
    main()

In [0]:
import lakebase

rows = lakebase.run_query("""
    SELECT
        id,
        document_id,
        chunk_index,
        LEFT(chunk_text, 150) AS chunk_preview,
        model_name,
        vector_dims(embedding) AS dimensions
    FROM weather_embeddings
    ORDER BY created_at DESC
    LIMIT 20
""")

for row in rows:
    print(row)

In [0]:
counts = lakebase.run_query("""
    SELECT
        COUNT(*) AS embedding_count,
        COUNT(DISTINCT document_id) AS document_count
    FROM weather_embeddings
""")

print(counts)

In [0]:
import lakebase

lakebase.run_write("""
CREATE INDEX IF NOT EXISTS idx_weather_embeddings_hnsw
ON weather_embeddings
USING hnsw (embedding vector_cosine_ops)
""")

print("HNSW index created")

In [0]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

query = "flash flood risk this weekend"

query_embedding = model.encode(query)

query_vector_str = (
    "["
    + ",".join(str(float(x)) for x in query_embedding)
    + "]"
)

print("Dimensions:", len(query_embedding))

In [0]:
results = lakebase.run_query(
    """
    SELECT
        d.id,
        d.location,
        d.source_type,
        d.headline,
        e.chunk_index,
        e.chunk_text,
        1 - (e.embedding <=> %s::vector) AS similarity
    FROM weather_embeddings e
    JOIN weather_documents d
        ON d.id = e.document_id
    ORDER BY e.embedding <=> %s::vector
    LIMIT %s
    """,
    (
        query_vector_str,
        query_vector_str,
        5,
    ),
)

for row in results:
    print("\n----------------")
    print("Location:", row["location"])
    print("Type:", row["source_type"])
    print("Headline:", row["headline"])
    print("Similarity:", row["similarity"])
    print("Text:", row["chunk_text"][:400])

In [0]:
%pip install flask --quiet

import importlib
import app
importlib.reload(app)

client = app.app.test_client()

print("=== Test Queries to Find Seattle ===")
print()

test_queries = [
    "Seattle weather forecast",
    "Pacific Northwest rain",
    "weather in Seattle",
]

for query in test_queries:
    print(f"Query: '{query}'")
    
    response = client.post(
        "/weather/search",
        json={"query": query, "top_k": 10}
    )
    
    data = response.get_json()
    locations = {}
    
    for result in data['results']:
        loc = result['location']
        if loc not in locations:
            locations[loc] = []
        locations[loc].append(result['similarity'])
    
    print(f"  Results by location:")
    for loc, sims in sorted(locations.items()):
        avg_sim = sum(sims) / len(sims)
        print(f"    • {loc}: {len(sims)} results (avg similarity: {avg_sim:.3f})")
    print()

print("\n=== Try Broad Query with More Results ===")
print()
print("Query: 'weather forecast'")

response = client.post(
    "/weather/search",
    json={"query": "weather forecast", "top_k": 20}
)

data = response.get_json()
locations = {}

for result in data['results']:
    loc = result['location']
    if loc not in locations:
        locations[loc] = []
    locations[loc].append(result['similarity'])

print(f"Found {len(data['results'])} total results:")
for loc, sims in sorted(locations.items()):
    avg_sim = sum(sims) / len(sims)
    print(f"  • {loc}: {len(sims)} results (avg similarity: {avg_sim:.3f})")

In [0]:
import sys
sys.path.append('/Workspace/Users/nikhita.nikki@gmail.com/databricks-lakebase-app-day-2')

import importlib
import app
importlib.reload(app)

client = app.app.test_client()

# Check embeddings count via API
response = client.post(
    "/weather/generate-embeddings",  
    json={}
)

data = response.get_json()
print(f"Embedding generation result: {data}")
print()
print("If embedded=0, all documents already have embeddings ✓")

In [0]:
import sys
sys.path.append('/Workspace/Users/nikhita.nikki@gmail.com/databricks-lakebase-app-day-2')
import lakebase

print("=== Dropping problematic vector index ===")
print()

# The ivfflat index needs 1000+ vectors to work properly
# With only ~28 vectors, it causes search issues
lakebase.run_write("""
DROP INDEX IF EXISTS idx_weather_embeddings_vector
""")

print("✓ Index dropped successfully")
print("  Sequential scan will be used instead (fine for small datasets)")

In [0]:
import sys
sys.path.append('/Workspace/Users/nikhita.nikki@gmail.com/databricks-lakebase-app-day-2')
import lakebase

print("=== Check All Locations in weather_documents ===")
locs = lakebase.run_query("""
SELECT location, COUNT(*) as count 
FROM weather_documents 
GROUP BY location 
ORDER BY location
""")
for loc in locs:
    print(f"  • {loc['location']}: {loc['count']} documents")

print("\n=== Check All Locations in weather_embeddings ===")
emb_locs = lakebase.run_query("""
SELECT d.location, COUNT(*) as count
FROM weather_embeddings e
JOIN weather_documents d ON d.id = e.document_id
GROUP BY d.location
ORDER BY d.location
""")
for loc in emb_locs:
    print(f"  • {loc['location']}: {loc['count']} embeddings")

print("\n=== Sample Seattle Documents ===")
seattle_docs = lakebase.run_query("""
SELECT id, headline, LEFT(narrative_text, 60) as preview
FROM weather_documents
WHERE location = 'Seattle, WA'
LIMIT 3
""")
for doc in seattle_docs:
    print(f"  {doc['headline']}: {doc['preview']}...")

In [0]:
response = client.post(
    "/weather/search",
    json={"query": ""}
)

print(response.status_code)
print(response.get_json())

In [0]:
response = client.post(
    "/weather/search",
    json={
        "query": "rain",
        "top_k": 1000,
    },
)

print(response.get_json()["top_k"])

In [0]:
import lakebase

rows = lakebase.run_query("SELECT 1 AS test")
print(rows)

In [0]:
rows = lakebase.run_query("""
SELECT table_name
FROM information_schema.tables
WHERE table_name = 'weather_embeddings'
""")

print(rows)